# LiteVulGNN: PHP Vulnerability Detection Dataset Pipeline

This notebook builds a graph dataset from legacy PHP source files and loads it for GNN training.

**Pipeline steps:**
1. **Tokenize** each `.php` file into a structural token-graph via `raw/extract_ast.php` (uses PHP's built-in tokenizer, no external dependencies).
2. **Vectorize** each token-graph into PyTorch Geometric tensors (`x`, `edge_index`) via `raw/graph_builder.py`.
3. **Build** the full dataset (`raw/build_dataset.py`), labeling files from `vulnerable/` as `y=1` and `secure/` as `y=0`, and save it to `processed/php_dataset.pt`.
4. **Load** the dataset with `raw/dataset.py`'s `PHPGraphDataset` and split it into train/val/test `DataLoader`s.

In [1]:
import sys, os, subprocess

# Make the raw/ folder (build_dataset.py, dataset.py, graph_builder.py) importable
RAW_DIR = os.path.join(os.getcwd(), 'raw')
if RAW_DIR not in sys.path:
    sys.path.insert(0, RAW_DIR)

# Sanity check: the PHP CLI is required to run extract_ast.php
php_check = subprocess.run(['php', '-v'], capture_output=True, text=True)
print(php_check.stdout.splitlines()[0] if php_check.returncode == 0 else "PHP CLI not found on PATH")

PHP 8.2.12 (cli) (built: Oct 24 2023 21:15:15) (ZTS Visual C++ 2019 x64)


In [4]:
from build_dataset import build_dataset_from_raw

# Tokenizes every .php file under vulnerable/ (y=1) and secure/ (y=0),
# converts each token-graph into a PyG Data object, and saves the list to processed/php_dataset.pt
build_dataset_from_raw()

Beginning Dataset Extraction...
Processing 10 files in c:\xampp\htdocs\CodeGraph\DataSet\vulnerable...
Processing 10 files in c:\xampp\htdocs\CodeGraph\DataSet\secure...
Processing Complete! Successfully generated 20 graph objects.
Dataset saved to c:\xampp\htdocs\CodeGraph\DataSet\processed\php_dataset.pt


In [5]:
import torch
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from dataset import PHPGraphDataset

# 1. Instantiate the Dataset (root = this notebook's folder, which holds processed/php_dataset.pt)
full_dataset = PHPGraphDataset(root=os.getcwd())
print(f"Total Graphs Loaded: {len(full_dataset)}")

# 2. Extract indices for Train/Val/Test Split (80% Train, 10% Val, 10% Test)
indices = list(range(len(full_dataset)))
labels = [full_dataset.get(i).y.item() for i in indices]

train_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)
val_idx, test_idx = train_test_split(
    test_idx, test_size=0.5, stratify=[labels[i] for i in test_idx], random_state=42
)

# 3. Subset the Dataset
train_dataset = [full_dataset[i] for i in train_idx]
val_dataset = [full_dataset[i] for i in val_idx]
test_dataset = [full_dataset[i] for i in test_idx]

# 4. Initialize DataLoaders for PyTorch
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Total Graphs Loaded: 20
Train batches: 1 | Val batches: 1 | Test batches: 1
